In [ ]:
import pandas as pd
import os

raw_dir = "../data/raw"

files = sorted([f for f in os.listdir(raw_dir) if f.endswith(".csv")])

print(f"Total daily files: {len(files)}")

df = pd.read_csv(os.path.join(raw_dir, files[0]))

print(f"shape: {df.shape}")

df.head()

Total daily files: 91
shape: (274660, 193)


,date,serial_number,model,capacity_bytes,failure,datacenter,cluster_id,vault_id,pod_id,pod_slot_num,...,smart_250_normalized,smart_250_raw,smart_251_normalized,smart_251_raw,smart_252_normalized,smart_252_raw,smart_254_normalized,smart_254_raw,smart_255_normalized,smart_255_raw
0,2024-01-01,WD-WX31DB48X22V,WDC WD60EFRX,6001175126016,0,sac0,0,1002,0,24.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-01-01,Z4D00WGP,ST6000DX000,6001175126016,0,sac0,0,1002,0,15.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-01-01,Z4D00YC6,ST6000DX000,6001175126016,0,sac0,0,1002,0,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-01-01,Z4D00YS2,ST6000DX000,6001175126016,0,sac0,0,1002,0,6.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-01-01,Z4D029JS,ST6000DX000,6001175126016,0,sac0,0,1002,0,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
#Check Faiure Rate

print("Failure value counts:")

print(df["failure"].value_counts())

print(f"Failure rate: {df['failure'].mean() * 100:.2f}%")

print("Failure is rare, this is a class imbalance problem we will handle in Day 3.")

Failure value counts:
failure
0    274657
1         3
Name: count, dtype: int64
Failure rate: 0.00%
Failure is rare, this is a class imbalance problem we will handle in Day 3.


In [6]:
# Understanding the 6 SMART columns

key_smart = {
    "smart_5_raw": "Reallocated Sectors - Physical damage indicator",
    "smart_9_raw": "Power-On Hours - Total hours the drive has been powered on",
    "smart_187_raw": "Reported Uncorrected Errors - Total count of uncorrected errors reported by the interface",
    "smart_188_raw": "Command Timeout - Total count of commands that have timed out",
    "smart_197_raw": "Current Pending Sector Count - Count of sectors pending reallocation",
    "smart_198_raw": "Uncorrected Sector Count - Total count of uncorrected errors"
}

print("Key SMART attributes for failure prediction:")
for col, desc in key_smart.items():
    present = col in df.columns
    status= "FOUND" if present else "MISSING"
    print(f"[{status}] {col}: {desc}")

Key SMART attributes for failure prediction:
[FOUND] smart_5_raw: Reallocated Sectors - Physical damage indicator
[FOUND] smart_9_raw: Power-On Hours - Total hours the drive has been powered on
[FOUND] smart_187_raw: Reported Uncorrected Errors - Total count of uncorrected errors reported by the interface
[FOUND] smart_188_raw: Command Timeout - Total count of commands that have timed out
[FOUND] smart_197_raw: Current Pending Sector Count - Count of sectors pending reallocation
[FOUND] smart_198_raw: Uncorrected Sector Count - Total count of uncorrected errors


In [9]:
#Loading all Q1 files together
from tqdm import tqdm
COLS = ["date", "serial_number", "model", "failure",
        "smart_5_raw", "smart_9_raw", "smart_187_raw",
        "smart_188_raw", "smart_197_raw", "smart_198_raw"]


dfs = []
for fname in tqdm(files, desc="Loading files"):
    df_temp = pd.read_csv(os.path.join(raw_dir, fname), usecols=COLS)
    dfs.append(df_temp)

df_full = pd.concat(dfs, ignore_index=True)

print(f"Full Dataset:")

print(f"Shape: {df_full.shape}")
print(f"Date Range: {df_full['date'].min()} to {df_full['date'].max()}")
print(f"Unique Drives: {df_full['serial_number'].nunique()}")

print(f"Total Failures: {df_full['failure'].sum()}")

print(f"Failure Rate: {df_full['failure'].mean() * 100:.4f}%")



Loading files: 100%|██████████| 91/91 [04:08<00:00,  2.74s/it]


Full Dataset:
Shape: (25189213, 10)
Date Range: 2024-01-01 to 2024-03-31
Unique Drives: 291331
Total Failures: 978
Failure Rate: 0.0039%


In [11]:
import subprocess

os.makedirs("../data/processed", exist_ok=True)
out_path = "../data/processed/q1_2024_combined.csv"
df_full.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Size: {os.path.getsize(out_path)/1e6:.1f} MB")

Saved: ../data/processed/q1_2024_combined.csv
Size: 1673.3 MB
